# Bets e consumo: painel UF × mês

**Pergunta:** o dinheiro que vai para apostas sai do consumo? Se sim, nas UFs onde o interesse por apostas cresceu
mais que a média, o volume de vendas do varejo e de serviços deveria crescer **menos**.

```
100·log(volume[uf, t]) = β · trends[uf, t−k] + γ · selic[t] × pobreza[uf] + α_uf + δ_t + ε
```

Mesmo desenho do `inadimplencia_painel.ipynb` (efeitos fixos de UF e de mês, controle Selic × pobreza, EP por UF e
wild bootstrap). As decisões comuns estão explicadas lá. Aqui ficam só as específicas do consumo.

- **Varejo** (PMC, tabela 8880): supermercados, vestuário, móveis, farmácia etc.
- **Varejo ampliado** (PMC, tabela 8881): varejo + veículos, material de construção e atacado alimentício.
- **Serviços** (PMS, tabela 5906): serviços prestados às famílias e às empresas.

Dados: `uv run python -m src.consumo.pmc_pms` e `uv run python -m src.exposicao.google_trends`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import pyfixest as pf

from src.config import PROCESSED
from src.exposicao.google_trends import NOME_UF
from src.painel import estimar, painel

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", "{:.3f}".format)

## Decisões

**Variável:** índice de **volume com ajuste sazonal** do IBGE, por UF. Volume, e não receita, porque queremos
quantidade consumida, já descontada a inflação do setor. O ajuste sazonal é o do próprio IBGE, feito UF a UF: o
efeito fixo de mês tira a sazonalidade nacional, mas não diferenças de sazonalidade entre UFs (ex.: turismo no Nordeste).

**Escala:** `100 · log(índice)`. Assim β é a **variação % do volume** quando o Trends da UF sobe 1 desvio-padrão.

**Defasagem:** ao contrário da inadimplência, que leva meses para aparecer, o deslocamento de consumo seria imediato:
o dinheiro apostado no mês deixa de ser gasto no mesmo mês. A especificação principal, **k = 0**, foi fixada antes de
rodar. Reportamos também k = 3 e 6, caso o efeito venha via aperto de crédito.

**Período:** jan/2022 a ago/2026, como no painel de inadimplência.

In [2]:
c = (pl.read_parquet(PROCESSED / "consumo_pmc_pms.parquet")
     .filter((pl.col("cod_uf") != "1") & pl.col("variavel").str.contains("ajuste sazonal") & pl.col("indice").str.contains("olume"))
     .with_columns(pl.col("uf_nome").replace_strict(NOME_UF).alias("uf"), (100 * pl.col("valor").log()).alias("y")))
PESQUISAS = {"pmc_varejo": "Varejo", "pmc_ampliado": "Varejo ampliado", "pms_servicos": "Serviços"}
paineis = {p: painel(c.filter(pl.col("pesquisa") == p).select("uf", "data", "y")).to_pandas() for p in PESQUISAS}
for p, d in paineis.items():
    print(f"{PESQUISAS[p]:16} {d['uf'].nunique()} UFs × {d['mes'].nunique()} meses = {len(d)} obs")

Varejo           27 UFs × 56 meses = 1512 obs
Varejo ampliado  27 UFs × 56 meses = 1512 obs
Serviços         27 UFs × 56 meses = 1512 obs


## 1. Resultado principal

In [3]:
res = pd.DataFrame([estimar(paineis[p], y="y", x=f"trends_l{k}", rotulo=f"{PESQUISAS[p]}, k = {k}")
                    for p in PESQUISAS for k in (0, 3, 6)]).set_index("modelo")
res

,β,ep,p,p_boot,n
modelo,,,,,
"Varejo, k = 0",1.276,0.992,0.209,0.235,1512
"Varejo, k = 3",1.765,0.988,0.086,0.099,1512
"Varejo, k = 6",1.677,1.140,0.153,0.132,1512
"Varejo ampliado, k = 0",-0.106,1.347,0.938,0.937,1512
"Varejo ampliado, k = 3",0.467,1.264,0.715,0.736,1512
"Varejo ampliado, k = 6",0.676,1.275,0.601,0.713,1512
"Serviços, k = 0",-1.335,1.755,0.454,0.485,1512
"Serviços, k = 3",-1.324,1.426,0.362,0.375,1512
"Serviços, k = 6",-0.987,1.059,0.360,0.349,1512


## 2. Placebo temporal

Trends 6 meses **à frente** junto com o Trends contemporâneo. Se o Trends futuro "explicar" o consumo de hoje, há
tendência comum (ex.: UF em expansão econômica que consome mais *e* passa a buscar mais apostas).

In [4]:
linhas = []
for p, d in paineis.items():
    fit = pf.feols("y ~ trends_l0 + trends_f6 + selic_x_pobreza | uf + mes", data=d.dropna(subset=["trends_f6"]), vcov={"CRV1": "uf_id"})
    t = fit.tidy()
    for v in ["trends_l0", "trends_f6"]:
        linhas.append({"pesquisa": PESQUISAS[p], "termo": v, "β": t.loc[v, "Estimate"], "ep": t.loc[v, "Std. Error"], "p": t.loc[v, "Pr(>|t|)"]})
pd.DataFrame(linhas)

,pesquisa,termo,β,ep,p
0,Varejo,trends_l0,1.186,0.951,0.223
1,Varejo,trends_f6,0.243,1.604,0.881
2,Varejo ampliado,trends_l0,0.446,1.017,0.665
3,Varejo ampliado,trends_f6,-0.878,1.802,0.630
4,Serviços,trends_l0,-0.881,1.076,0.420
5,Serviços,trends_f6,-0.838,1.978,0.675


## 3. Tamanho do efeito descartado

Como no painel de inadimplência, os efeitos fixos absorvem a maior parte da variação do Trends. Reescalamos o IC95 de
β (k = 0) para um choque de **1 DP residual** (o tamanho típico de uma UF se descolando da média nacional).

In [5]:
d0 = paineis["pmc_varejo"]
res_x = d0["trends_l0"] - pf.feols("trends_l0 ~ 1 | uf + mes", data=d0).predict()
dp = res_x.std()
linhas = []
for p in PESQUISAS:
    b, ep = res.loc[f"{PESQUISAS[p]}, k = 0", "β"], res.loc[f"{PESQUISAS[p]}, k = 0", "ep"]
    linhas.append({"pesquisa": PESQUISAS[p], "β por 1 DP residual (%)": b * dp,
                   "IC95 inf (%)": (b - 1.96 * ep) * dp, "IC95 sup (%)": (b + 1.96 * ep) * dp})
print(f"DP residual do Trends: {dp:.3f}")
pd.DataFrame(linhas).set_index("pesquisa")

DP residual do Trends: 0.221


,β por 1 DP residual (%),IC95 inf (%),IC95 sup (%)
pesquisa,,,
Varejo,0.282,-0.147,0.712
Varejo ampliado,-0.023,-0.607,0.560
Serviços,-0.295,-1.056,0.465


## Leitura

**Não há sinal de que as apostas tenham deslocado consumo entre UFs.**
- **Varejo:** β é **positivo** (o contrário do deslocamento), mas não significativo (p entre 0,1 e 0,24). Para um
  choque típico, o IC95 vai de cerca de −0,15% a +0,7% do volume. Uma queda de consumo maior que ~0,15% está descartada.
- **Varejo ampliado:** β ~0 em todas as defasagens.
- **Serviços:** β negativo (na direção do deslocamento), mas impreciso. O IC95 vai de cerca de −1,1% a +0,5%,
  então não dá para descartar um efeito moderado.

**Por que o varejo poderia até subir?** O sinal positivo não significativo não deve ser interpretado, mas uma
explicação plausível é que prêmios e "giro" de apostas devolvem liquidez a parte dos apostadores. A perda líquida
(GGR, ~R$ 3,3 bi/mês no mercado regulado em 2026) é pequena perto do consumo das famílias, da ordem de R$ 600–700 bi/mês
(~63% do PIB). Mesmo que toda ela saísse do consumo, seria ~0,5%, no limite do que este desenho consegue detectar.

**Esse é o ponto principal: escala.** A perda líquida com bets é da ordem de 0,5% do consumo. Um efeito desse tamanho,
espalhado por toda a UF, fica abaixo da resolução de um painel de 27 UFs. O nulo é **coerente** com o tamanho do setor,
não evidência de que as apostas não afetam o orçamento de quem aposta.

**Mesmas ressalvas do painel de inadimplência:** o Trends mede busca, não gasto (atenua β); o efeito fixo de mês
descarta o efeito nacional; o teste mais forte é o choque da MP 1.394, quando a PMC de out–dez/2026 sair.